# Feature Engineering — Temporal Metabolic Dynamics

This notebook creates temporal features to model delayed metabolic effects.

Glucose regulation is a dynamic process:
- Carbohydrates impact glucose with delay.
- Insulin acts over time.
- Physical activity affects glucose gradually.

Therefore, we construct rolling and trend-based features to capture these delayed effects.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../data/HUPA0016P.csv", sep=";")

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

df.head()

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2019-03-27 16:15:00,55.000000,5.866560,94.265152,0.0,0.069167,0.0,0.0
1,2019-03-27 16:20:00,55.333333,5.866560,92.787879,0.0,0.069167,0.0,0.0
2,2019-03-27 16:25:00,55.666667,5.773440,89.572464,0.0,0.069167,0.0,0.0
3,2019-03-27 16:30:00,56.000000,22.535040,112.895105,278.0,0.069167,0.0,0.0
4,2019-03-27 16:35:00,64.666667,34.454401,125.904762,491.0,0.069167,0.0,0.0


## Why Feature Engineering

Raw variables do not capture temporal dependencies nor physiological delays.

Examples:
- A meal affects glucose for the next 30–120 minutes.
- An insulin bolus lowers glucose progressively over time.
- Physical activity shifts glucose trends with a delay.

To model these effects we create features using a 5‑minute sampling baseline.

In [15]:
# Glucose memory
df["glucose_lag1"] = df["glucose"].shift(1)
df["glucose_lag2"] = df["glucose"].shift(2)
df["glucose_ma_30"] = df["glucose"].rolling(6).mean()

# Interventions
# Carbs acumulated every 45 min
df["carbs_45m"] = df["carb_input"].rolling(9).sum()
# Bolus rolling mean every 45 min
df["bolus_45m"] = df["bolus_volume_delivered"].rolling(9).sum()
# Steps last 30 min
df["steps_30m"] = df["steps"].rolling(6).sum()

# Other physiological variables
# Calories burned last 30 min
df["calories_30m"] = df["calories"].rolling(6).sum()
# Heart rate rolling mean every 30 min
df["heart_rate_ma_30"] = df["heart_rate"].rolling(6).mean()
# Basal rate rolling mean every 45 min
df["basal_rate_45m"] = df["basal_rate"].rolling(9).sum()

# Circadian
df["hour"] = df["time"].dt.hour

In [16]:
# Verify new columns (actualizados)
cols = [
    "glucose",
    "glucose_lag1",
    "glucose_lag2",
    "glucose_ma_30",
    "carbs_45m",
    "steps_30m",
    "bolus_45m",
    "calories_30m",
    "heart_rate_ma_30",
    "basal_rate_45m",
    "hour",
]

df[cols].head(15)

,glucose,glucose_lag1,glucose_lag2,glucose_ma_30,carbs_45m,steps_30m,bolus_45m,calories_30m,heart_rate_ma_30,basal_rate_45m,hour
0,55.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16
1,55.333333,55.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16
2,55.666667,55.333333,55.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16
3,56.000000,55.666667,55.333333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16
4,64.666667,56.000000,55.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16
5,73.333333,64.666667,56.000000,60.000000,NaN,798.0,NaN,90.140160,102.913422,NaN,16
6,82.000000,73.333333,64.666667,64.500000,NaN,798.0,NaN,89.674560,102.304492,NaN,16
7,94.666667,82.000000,73.333333,71.055556,NaN,798.0,NaN,89.395200,101.002164,NaN,16
8,107.333333,94.666667,82.000000,79.666667,0.0,922.0,0.0,89.208960,99.868696,0.6225,16
9,120.000000,107.333333,94.666667,90.333333,0.0,651.0,0.0,73.844160,95.024390,0.6225,17


## Engineered Features Summary

The following features were constructed (names reflect rolling windows based on 5‑minute sampling):

- `glucose_lag1`, `glucose_lag2` → immediate past glucose values (short-term memory).
- `glucose_ma_30` → 30‑minute moving average of glucose (rolling 6).
- `carbs_45m` → carbohydrates accumulated over the last 45 minutes (rolling 9).
- `bolus_45m` → insulin delivered accumulated over the last 45 minutes (rolling 9).
- `steps_30m` → steps accumulated over the last 30 minutes (rolling 6).
- `calories_30m` → calories burned in the last 30 minutes (rolling 6).
- `heart_rate_ma_30` → 30‑minute average heart rate (rolling 6).
- `basal_rate_45m` → basal insulin accumulated over 45 minutes (rolling 9).
- `hour` → hour of day for circadian encoding.

These engineered features introduce explicit temporal memory and physiologically‑relevant windows. They improve a model's ability to learn delayed causal effects, which is essential for realistic metabolic prediction and simulation.